## Data prep: Füge composites files zusammen

In [1]:
from pathlib import Path
import datetime as dt

import geopandas as gpd
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import zarr
import pandas as pd

In [11]:
erstelldatum = dt.date.today()
lags = [-4,-3,-2,-1,0] # [-6, -5, -4, -3, -2, -1, 0]
region_ziel = "Norddeutschland"  # region der ziel variablen
region_prec = "EUROPE"  # welche Daten werden als precursor verwendet
timeperiod = "1979-2025"  # zeitraum der analysierten daten
quant = 0.9  # welches quantile => def extreme event
extreme_event = "h"  # l=low 0.1, h=high 0.9

In [12]:
var_energy = 'SPV'
var_atm = 'relative_humidity'  # 'mean_sea_level_pressure', 'geopotential', 'wind_speed', 'relative_humidity', 'total_column_water_vapour'

In [13]:
df0 = xr.open_dataset(f'data/domino_output_coords_weatherbench/{var_energy}/{var_energy}-{extreme_event}_{region_ziel}_{var_atm}_{region_prec}_1979_lags_-4_-2_-1_lagged-means_coords_deseasonalized_DJF_anom.nc')

In [14]:
df1 = xr.open_dataset(f'data/domino_output_coords_weatherbench/{var_energy}/{var_energy}-{extreme_event}_{region_ziel}_{var_atm}_{region_prec}_1979_lags_-6_-3_0_lagged-means_coords_deseasonalized_DJF_anom.nc')

In [15]:
# entferne lag 0 aus df0
df0 = df0.sel(lag=[-4, -2, -1])
# entferne lag -6 aus df1
df1 = df1.sel(lag=[-3, 0])

In [16]:
df_all = xr.concat([df0, df1], dim='lag')

In [17]:
# sortiere die lags in df_all
df_all = df_all.sortby('lag')

In [18]:
df_all

<xarray.Dataset> Size: 3MB
Dimensions:            (lag: 5, latitude: 289, longitude: 561)
Coordinates:
  * lag                (lag) int64 40B -4 -3 -2 -1 0
  * latitude           (latitude) float32 1kB 84.0 83.75 83.5 ... 12.25 12.0
  * longitude          (longitude) float32 2kB 270.0 270.2 270.5 ... 49.75 50.0
Data variables:
    relative_humidity  (lag, latitude, longitude) float32 3MB 0.0005595 ... -...

In [19]:
# df_all abspeichern
df_all.to_netcdf(f'data/domino_output_coords_weatherbench/{var_energy}/{var_energy}-{extreme_event}_{region_ziel}_{var_atm}_{region_prec}_1979_lags_{"_".join(map(str, lags))}_lagged-means_coords_deseasonalized_DJF_anom.nc')